# ROI Crossnobis RSA — group results

Crossnobis RDMs over the 8 stimulus identities from GLMsingle type-D cue betas, produced by
`run_rsa_roi.py`. Masks: whole brain, visual cortex, fusiform, vmPFC, striatum.

The question: *are stimuli with similar objective value represented more similarly?* Unlike
the high/low classifier, this makes a prediction identity and category cannot mimic — the
three pairs of stimuli that **share** a reward level should be extra-similar despite being
different images in different categories.

Prerequisites, both already run:
[`rsa_design_checks.ipynb`](rsa_design_checks.ipynb) (what the design allows) ·
[`crossnobis_validation.ipynb`](crossnobis_validation.ipynb) (that the distance is correct)

## Reading the numbers

Established in [`rsa_design_checks.ipynb`](rsa_design_checks.ipynb) (n=62):

- **`subset='nonfigure'` is primary.** Values {1,5} always sit on `figure` (62/62), so the
  all-8 version cannot separate extreme value from figure-vs-rest.
- **Frequency stays in the model** — corr(value, frequency) = −0.346, fixed for everyone.
- **The same-value contrast is conservative**: those pairs carry the maximal |Δfreq|.

Conventions:

- Model RDMs are *dis*similarity predictions → **positive β = represented**.
- `contrast_value` = mean(diff-value) − mean(same-value) → **positive = value coding**.
- β are standardised partial weights (RDM and predictors z-scored), comparable across
  subjects and ROIs. Tested against 0, which crossnobis's unbiasedness licenses.
- No identity regressor: every off-diagonal cell is different-identity, so identity *is*
  the intercept.

**Status:** _fill in after running — n subjects, date._

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

DERIV_DIR = Path("/Users/hugofluhr/phd_local/data/LearningHabits/dev_sample/bids_dataset/derivatives")
RSA_DIR = DERIV_DIR / "rsa"
RSA_SHUFFLED_DIR = DERIV_DIR / "rsa_shuffled"   # negative control; may be absent

MASKS = ["wholebrain", "visualcortex", "fusiform", "vmpfc", "striatum"]
MASK_LABELS = {"wholebrain": "Whole brain", "visualcortex": "Visual cortex",
               "fusiform": "Fusiform", "vmpfc": "vmPFC", "striatum": "Striatum"}
COLORS = {"wholebrain": "#4C72B0", "visualcortex": "#DD8452", "fusiform": "#937860",
          "vmpfc": "#55A868", "striatum": "#C44E52"}

SCOPES = ["pooled", "learning1", "learning2", "test"]
RUN_SCOPES = ["learning1", "learning2", "test"]
MODEL_TERMS = ["category", "value", "frequency"]
TERM_COLORS = {"category": "#8172B3", "value": "#C44E52", "frequency": "#64B5CD"}

# Primary readouts. 'nonfigure' is the counterbalanced test (values {1,5} always sit on
# the figure category, so the all-8 version cannot separate extreme value from
# figure-vs-rest); 'objective' is the fixed reward level, 'rl' the model-derived Q.
PRIMARY_SUBSET = "nonfigure"
PRIMARY_MODEL = "objective"

In [ ]:
def load_results(rsa_dir):
    """Concatenate every subject's long-format RSA results CSV."""
    files = sorted(Path(rsa_dir).glob("sub-*/sub-*_rsa_results.csv"))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def load_rdms(rsa_dir, mask, scope):
    """{subject: 8x8 crossnobis RDM} for one mask/scope, rows ordered as stim_names."""
    out = {}
    for f in sorted(Path(rsa_dir).glob(f"sub-*/sub-*_rsa_rdm_{mask}_{scope}.npy")):
        out[f.parent.name] = np.load(f)
    return out


def load_model_rdms(rsa_dir):
    """{subject: npz} of per-subject model RDMs and per-stimulus properties."""
    out = {}
    for f in sorted(Path(rsa_dir).glob("sub-*/sub-*_rsa_model_rdms.npz")):
        out[f.parent.name] = np.load(f, allow_pickle=True)
    return out


def group_test(values):
    """One-sample t-test and Wilcoxon against 0, plus mean/sem. Valid against 0 because
    crossnobis distances are unbiased under the null, so the regression coefficients are
    too (no need for a permutation reference for the location test)."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if len(v) < 3:
        return dict(n=len(v), mean=np.nan, sem=np.nan, t=np.nan, p=np.nan, p_wilcoxon=np.nan)
    t, p = stats.ttest_1samp(v, 0)
    try:
        _, pw = stats.wilcoxon(v)
    except ValueError:
        pw = np.nan
    return dict(n=len(v), mean=v.mean(), sem=v.std(ddof=1) / np.sqrt(len(v)),
                t=float(t), p=float(p), p_wilcoxon=float(pw))


def summarise(df, column, scope="pooled", subset=PRIMARY_SUBSET, model=PRIMARY_MODEL):
    """Group test of `column` per mask, for one (scope, subset, model) slice."""
    sel = df[(df.scope == scope) & (df["subset"] == subset) & (df.model == model)]
    rows = []
    for mask in MASKS:
        vals = sel.loc[sel["mask"] == mask, column].values
        rows.append(dict(mask=mask, **group_test(vals)))
    return pd.DataFrame(rows).set_index("mask")


def stars(p):
    if not np.isfinite(p):
        return ""
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""


df = load_results(RSA_DIR)
df_shuf = load_results(RSA_SHUFFLED_DIR)
model_rdms = load_model_rdms(RSA_DIR)

print(f"real:     {df.subject.nunique() if len(df) else 0} subjects, {len(df)} rows")
print(f"shuffled: {df_shuf.subject.nunique() if len(df_shuf) else 0} subjects")
if len(df):
    print("scopes:", sorted(df.scope.unique()), "| subsets:", sorted(df['subset'].unique()),
          "| models:", sorted(df.model.unique()))
    print("within-run split:", df.within_run_split.unique())
    print("\nvoxels per mask (median):")
    print(df.groupby("mask").n_voxels.median().reindex(MASKS).to_string())

## 1. Model RDMs

Re-derived from the saved npz files, i.e. from the data actually processed. Should match
`rsa_design_checks.ipynb`.

In [ ]:
# Re-derive the design facts from the processed data rather than trusting the docstring.
iu = np.triu_indices(8, 1)
rows, assignments = [], set()
for sub, npz in model_rdms.items():
    cat, val, frq = npz["category"], npz["value"], npz["frequency"]
    c, v, f = cat[iu], val[iu], frq[iu]
    nf = npz["non_figure"]
    names, values = npz["stim_names"], npz["stim_value"]
    assignments.add(tuple(zip(map(str, names), values.astype(int))))
    same_v, cross = v == 0, c > 0
    rows.append(dict(
        subject=sub,
        r_cat_val=np.corrcoef(c, v)[0, 1], r_cat_frq=np.corrcoef(c, f)[0, 1],
        r_val_frq=np.corrcoef(v, f)[0, 1],
        n_nonfigure=int(nf.sum()), n_same_value=int(same_v.sum()),
        same_value_all_cross=bool(cross[same_v].all()),
        dfrq_same_value=f[same_v].mean(), dfrq_diff_value=f[~same_v & cross].mean(),
        extreme_cat="/".join(sorted(set(np.asarray(npz["stim_cat"])[np.isin(values, [1, 5])]))),
    ))
mr = pd.DataFrame(rows)

print(f"distinct image->value assignments across {len(mr)} subjects: {len(assignments)}")
print(f"category holding values {{1,5}}: {mr.extreme_cat.value_counts().to_dict()}")
print(f"all same-value pairs cross-category: {mr.same_value_all_cross.all()}")
print(f"\nmodel RDM intercorrelations (min .. max across subjects):")
for c in ["r_cat_val", "r_cat_frq", "r_val_frq"]:
    print(f"  {c:11s} {mr[c].min():+.3f} .. {mr[c].max():+.3f}")
print(f"\nmean |dfreq| on same-value pairs   : {mr.dfrq_same_value.mean():.3f}")
print(f"mean |dfreq| on diff-value x-cat   : {mr.dfrq_diff_value.mean():.3f}"
      "   <- frequency runs AGAINST the value hypothesis in contrast_value")

In [ ]:
# One example subject's model RDMs. Non-negative dissimilarity predictions -> sequential
# single-hue map (see the signed crossnobis RDMs below for why those differ).
ex_sub = sorted(model_rdms)[0]
npz = model_rdms[ex_sub]
labels = [f"{n}\n(v={int(v)}, f={int(f):+d})" for n, v, f
          in zip(npz["stim_names"], npz["stim_value"], npz["stim_frequency"])]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
for ax, term in zip(axes, MODEL_TERMS):
    m = npz[term]
    im = ax.imshow(m, cmap="Purples", vmin=0)
    ax.set_title(f"{term}   ({ex_sub})", fontsize=11)
    ax.set_xticks(range(8)); ax.set_yticks(range(8))
    ax.set_xticklabels(npz["stim_names"], rotation=90, fontsize=7, color="#444444")
    ax.set_yticklabels(labels, fontsize=7, color="#444444")
    fig.colorbar(im, ax=ax, fraction=0.046, label="model dissimilarity")
fig.suptitle("Model RDMs — one subject (the value map is counterbalanced across subjects)",
             fontsize=12)
fig.tight_layout()
plt.show()

# Intercorrelation spread across subjects. corr(cat,val) and corr(val,frq) are fixed by
# the design; corr(cat,frq) is the only one that varies, which is what lets the
# regression separate the category and frequency terms at the group level.
fig, ax = plt.subplots(figsize=(6.5, 3.6))
pairs = [("r_cat_val", "category ~ value"), ("r_cat_frq", "category ~ frequency"),
         ("r_val_frq", "value ~ frequency")]
for i, (col, lab) in enumerate(pairs):
    ax.scatter(mr[col], np.full(len(mr), i) + np.random.uniform(-.09, .09, len(mr)),
               s=18, color="#4C72B0", alpha=.55, edgecolor="none", zorder=3)
ax.axvline(0, color="#999999", linewidth=1, zorder=1)
ax.set_yticks(range(len(pairs))); ax.set_yticklabels([p[1] for p in pairs], color="#444444")
ax.set_xlabel("Pearson r between model RDM vectors (28 cells)", color="#444444")
ax.set_title("Model RDMs are only mildly correlated — but corr(cat,frq) is the\n"
             "only one that varies across subjects", fontsize=11)
ax.grid(axis="x", alpha=.25, zorder=0); ax.set_axisbelow(True)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
fig.tight_layout()
plt.show()

## 2. Group RDMs

Two views, because subjects have different image → value mappings:

- **image space** — structure here is visual (category/identity); value is scrambled by the
  counterbalancing. Sanity check: category blocks should show in visual cortex.
- **value space** — each subject's RDM reordered by *their* stimulus values before
  averaging. Value structure survives this; idiosyncratic image similarity doesn't.

Crossnobis is signed, so the maps are diverging with a neutral midpoint at 0.

In [ ]:
def _z_rdm(r):
    v = r[iu]
    return (r - v.mean()) / v.std() if v.std() > 0 else r - v.mean()


def group_rdm(mask, scope="pooled", space="image"):
    """Mean z-scored RDM across subjects, in image space or value space.

    Each subject's RDM is z-scored over its off-diagonal cells first so that subjects
    with globally larger distances do not dominate the average.
    """
    mats = load_rdms(RSA_DIR, mask, scope)
    stacked, order_labels = [], None
    for sub, r in mats.items():
        npz = model_rdms[sub]
        if space == "image":
            idx = np.argsort(npz["stim_names"])
            labs = [str(s) for s in npz["stim_names"][idx]]
        elif space == "value":
            # sort by (value, frequency) so every subject's rows line up by value
            idx = np.lexsort((npz["stim_frequency"], npz["stim_value"]))
            labs = [f"v={int(v)}, f={int(f):+d}" for v, f in
                    zip(npz["stim_value"][idx], npz["stim_frequency"][idx])]
        else:
            raise ValueError(space)
        if order_labels is None:
            order_labels = labs
        elif space == "value" and labs != order_labels:
            raise ValueError(f"{sub}: value ordering differs from the first subject — "
                             f"{labs} vs {order_labels}")
        stacked.append(_z_rdm(r)[np.ix_(idx, idx)])
    if not stacked:
        return None, None, 0
    return np.mean(stacked, axis=0), order_labels, len(stacked)


for space, blurb in [("image", "image space — structure here is visual, value is scrambled"),
                     ("value", "value space — reordered per subject by stimulus value")]:
    fig, axes = plt.subplots(1, len(MASKS), figsize=(4 * len(MASKS), 4.3))
    for ax, mask in zip(np.atleast_1d(axes), MASKS):
        g, labs, n = group_rdm(mask, "pooled", space)
        if g is None:
            ax.set_visible(False)
            continue
        lim = np.abs(g[iu]).max()
        im = ax.imshow(g, cmap="RdBu_r", vmin=-lim, vmax=lim)   # diverging, 0 = neutral
        ax.set_title(f"{MASK_LABELS[mask]}  (n={n})", fontsize=11)
        ax.set_xticks(range(8)); ax.set_yticks(range(8))
        ax.set_xticklabels(labs, rotation=90, fontsize=6.5, color="#444444")
        ax.set_yticklabels(labs, fontsize=6.5, color="#444444")
        fig.colorbar(im, ax=ax, fraction=0.046, label="mean z-scored crossnobis")
    fig.suptitle(f"Group RDMs, pooled over runs — {blurb}", fontsize=12)
    fig.tight_layout()
    plt.show()

## 3. Group model fits — pooled over runs

`RDM ~ 1 + category + value + frequency`, coefficients tested against 0 across subjects.
Shown for both subsets; `nonfigure` is the one to read.

In [ ]:
def beta_table(subset, model=PRIMARY_MODEL, scope="pooled", frame=None):
    frame = df if frame is None else frame
    out = {}
    for term in MODEL_TERMS:
        s = summarise(frame, f"beta_{term}", scope=scope, subset=subset, model=model)
        out[term] = s
    return out


for subset in ["nonfigure", "all"]:
    tabs = beta_table(subset)
    print(f"\n{'='*84}\nsubset = {subset!r}   scope = 'pooled'   model = {PRIMARY_MODEL!r}"
          f"   (n = {tabs['category']['n'].max():.0f})\n{'='*84}")
    print(f"{'mask':14s}" + "".join(f"{t:>22s}" for t in MODEL_TERMS))
    for mask in MASKS:
        cells = []
        for term in MODEL_TERMS:
            r = tabs[term].loc[mask]
            cells.append(f"{r['mean']:+.3f}±{r['sem']:.3f}{stars(r['p']):<3s}")
        print(f"{MASK_LABELS[mask]:14s}" + "".join(f"{c:>22s}" for c in cells))

# Grouped bars: one group per mask, one bar per model term (fixed hue order, never cycled)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, subset in zip(axes, ["nonfigure", "all"]):
    tabs = beta_table(subset)
    x = np.arange(len(MASKS)); w = 0.26
    for j, term in enumerate(MODEL_TERMS):
        s = tabs[term].reindex(MASKS)
        off = (j - 1) * w
        ax.bar(x + off, s["mean"], width=w - 0.02, color=TERM_COLORS[term],
               label=term if ax is axes[0] else None, zorder=3)
        ax.errorbar(x + off, s["mean"], yerr=s["sem"], fmt="none", color="#333333",
                    capsize=2.5, linewidth=1, zorder=4)
        for xi, (m, p) in enumerate(zip(s["mean"], s["p"])):
            if stars(p):
                ax.text(xi + off, m + np.sign(m) * (s["sem"].iloc[xi] + .012), stars(p),
                        ha="center", va="bottom" if m >= 0 else "top",
                        fontsize=9, color="#333333")
    ax.axhline(0, color="#555555", linewidth=1, zorder=2)
    ax.set_xticks(x); ax.set_xticklabels([MASK_LABELS[m] for m in MASKS], rotation=20,
                                         ha="right", color="#444444")
    n = int(tabs["category"]["n"].max())
    ax.set_title(f"subset = {subset}"
                 f"{'  (primary — counterbalanced)' if subset=='nonfigure' else '  (figure stimuli carry values 1/5)'}"
                 f"   n={n}", fontsize=11)
    ax.grid(axis="y", alpha=.25, zorder=0); ax.set_axisbelow(True)
    for s_ in ("top", "right"):
        ax.spines[s_].set_visible(False)
axes[0].set_ylabel("standardised partial β  (positive = represented)", color="#444444")
axes[0].legend(frameon=False, loc="upper left", title="model RDM")
fig.suptitle("Group RSA model fits, pooled over runs — objective value", fontsize=12)
fig.tight_layout()
plt.show()

## 4. The targeted contrast

Among pairs that are all different-identity and all different-category: are those sharing a
reward level more similar? This is the readout that most directly escapes the identity
confound. Only 3 same-value cells per subject, so the group test is the only meaningful
level.

In [ ]:
con = summarise(df, "contrast_value", scope="pooled", subset=PRIMARY_SUBSET)
print(f"contrast_value — pooled, subset={PRIMARY_SUBSET!r}  (positive = value coding)\n")
print(f"{'mask':14s}{'n':>4s}{'mean':>9s}{'sem':>8s}{'t':>8s}{'p':>10s}{'p(wilcox)':>12s}")
for mask in MASKS:
    r = con.loc[mask]
    print(f"{MASK_LABELS[mask]:14s}{r['n']:>4.0f}{r['mean']:>+9.3f}{r['sem']:>8.3f}"
          f"{r['t']:>8.2f}{r['p']:>10.4f}{stars(r['p']):<3s}{r['p_wilcoxon']:>9.4f}")

# Per-subject distribution behind each group mean — never show only the bar.
fig, ax = plt.subplots(figsize=(8.5, 4.6))
sel = df[(df.scope == "pooled") & (df["subset"] == PRIMARY_SUBSET) & (df.model == PRIMARY_MODEL)]
for i, mask in enumerate(MASKS):
    v = sel.loc[sel["mask"] == mask, "contrast_value"].dropna().values
    ax.scatter(np.full(len(v), i) + np.random.uniform(-.13, .13, len(v)), v,
               s=16, color=COLORS[mask], alpha=.45, edgecolor="none", zorder=3)
    r = con.loc[mask]
    ax.plot([i - .28, i + .28], [r["mean"]] * 2, color="#222222", linewidth=2, zorder=5)
    ax.errorbar(i, r["mean"], yerr=r["sem"], fmt="none", color="#222222",
                capsize=4, linewidth=1.4, zorder=5)
    if stars(r["p"]):
        ax.text(i, v.max() if len(v) else 0, stars(r["p"]), ha="center", va="bottom",
                fontsize=11, color="#333333")
ax.axhline(0, color="#555555", linewidth=1, zorder=2)
ax.set_xticks(range(len(MASKS)))
ax.set_xticklabels([MASK_LABELS[m] for m in MASKS], rotation=20, ha="right", color="#444444")
ax.set_ylabel("mean(diff-value) − mean(same-value)\ncross-category cells, z-scored RDM",
              color="#444444")
ax.set_title("Same-reward-level stimuli: more similar? (pooled, non-figure subset)\n"
             "positive = value coding; dots are subjects, bar is the group mean ± sem",
             fontsize=11)
ax.grid(axis="y", alpha=.25, zorder=0); ax.set_axisbelow(True)
for s_ in ("top", "right"):
    ax.spines[s_].set_visible(False)
fig.tight_layout()
plt.show()

## 5. Learning dynamics

Per-run RDMs (within-run 2-fold CV). Look for value structure rising learning1 → learning2,
and persisting into `test`, which has **no feedback** — persistence there is the habit-relevant
claim.

These rest on half the trials and 2 folds instead of 3, so they are much noisier than pooled.
`crossnobis_validation.ipynb` §4 shows the two split modes diverge badly at low SNR — treat a
marginal result here as provisional until it reproduces under `--within-run-split blocked`.

In [ ]:
def dynamics(column, model=PRIMARY_MODEL, subset=PRIMARY_SUBSET):
    """(mask x run_scope) group mean/sem/p of `column`."""
    out = {}
    for mask in MASKS:
        out[mask] = {sc: group_test(
            df[(df.scope == sc) & (df["subset"] == subset) & (df.model == model) &
               (df["mask"] == mask)][column].values) for sc in RUN_SCOPES}
    return out


for column, model, title in [
        ("beta_value", "objective", "β(objective value)"),
        ("beta_value", "rl", "β(RL Q-value)"),
        ("contrast_value", "objective", "same-value contrast")]:
    d = dynamics(column, model=model)
    print(f"\n{title}  —  subset={PRIMARY_SUBSET!r}")
    print(f"{'mask':14s}" + "".join(f"{s:>20s}" for s in RUN_SCOPES))
    for mask in MASKS:
        cells = [f"{d[mask][s]['mean']:+.3f}±{d[mask][s]['sem']:.3f}{stars(d[mask][s]['p']):<3s}"
                 for s in RUN_SCOPES]
        print(f"{MASK_LABELS[mask]:14s}" + "".join(f"{c:>20s}" for c in cells))

# One line per ROI across the three runs. Color follows the ROI (the entity), not rank.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharex=True)
panels = [("beta_value", "objective", "β(objective value)"),
          ("beta_value", "rl", "β(RL Q-value)"),
          ("contrast_value", "objective", "same-value contrast")]
x = np.arange(len(RUN_SCOPES))
for ax, (column, model, title) in zip(axes, panels):
    d = dynamics(column, model=model)
    for mask in MASKS:
        m = np.array([d[mask][s]["mean"] for s in RUN_SCOPES])
        e = np.array([d[mask][s]["sem"] for s in RUN_SCOPES])
        ax.plot(x, m, marker="o", markersize=6, linewidth=2, color=COLORS[mask],
                label=MASK_LABELS[mask] if ax is axes[0] else None, zorder=3)
        ax.fill_between(x, m - e, m + e, color=COLORS[mask], alpha=.15, zorder=2)
    ax.axhline(0, color="#555555", linewidth=1, zorder=1)
    ax.axvline(1.5, color="#BBBBBB", linewidth=1, linestyle=":", zorder=1)
    ax.text(1.52, ax.get_ylim()[1], " no feedback →", fontsize=8, color="#777777",
            va="top", ha="left")
    ax.set_xticks(x); ax.set_xticklabels(RUN_SCOPES, color="#444444")
    ax.set_title(title, fontsize=11)
    ax.grid(axis="y", alpha=.25, zorder=0); ax.set_axisbelow(True)
    for s_ in ("top", "right"):
        ax.spines[s_].set_visible(False)
axes[0].set_ylabel("group mean ± sem", color="#444444")
axes[0].legend(frameon=False, fontsize=9, loc="best")
fig.suptitle("Learning dynamics — per-run RDMs (within-run 2-fold CV; noisier than pooled)",
             fontsize=12)
fig.tight_layout()
plt.show()

## 6. Negative control — shuffled labels

End-to-end version of `crossnobis_validation.ipynb` §2, on real data. Everything should
collapse to ~0. Generate with:

```bash
SHUFFLE_SEED=1 OUTPUT_DIR=.../derivatives/rsa_shuffled bash multivariate/submit_rsa_roi.sh
```

Also checks `rdm_mean` straddles 0 under the shuffle (the unbiasedness property on real
data) and that the RDM diagonal is exactly 0.

In [ ]:
if not len(df_shuf):
    print(f"No shuffled control found at {RSA_SHUFFLED_DIR} — see the cell above to generate it.")
else:
    cols = [f"beta_{t}" for t in MODEL_TERMS] + ["contrast_value"]
    print(f"real (n={df.subject.nunique()})  vs  shuffled (n={df_shuf.subject.nunique()})"
          f"   pooled, subset={PRIMARY_SUBSET!r}\n")
    print(f"{'mask':14s}{'term':16s}{'real':>18s}{'shuffled':>18s}")
    for mask in MASKS:
        for c in cols:
            r = summarise(df, c, subset=PRIMARY_SUBSET).loc[mask]
            s = summarise(df_shuf, c, subset=PRIMARY_SUBSET).loc[mask]
            print(f"{MASK_LABELS[mask]:14s}{c.replace('beta_',''):16s}"
                  f"{r['mean']:>+13.3f}{stars(r['p']):<5s}"
                  f"{s['mean']:>+13.3f}{stars(s['p']):<5s}")

    fig, ax = plt.subplots(figsize=(9.5, 4.6))
    w = .38
    ticks, labs = [], []
    for i, mask in enumerate(MASKS):
        for j, c in enumerate(cols):
            k = i * len(cols) + j
            r = summarise(df, c, subset=PRIMARY_SUBSET).loc[mask]
            s = summarise(df_shuf, c, subset=PRIMARY_SUBSET).loc[mask]
            ax.bar(k - w/2, r["mean"], width=w - .02, color=COLORS[mask], zorder=3,
                   label="real" if k == 0 else None)
            ax.bar(k + w/2, s["mean"], width=w - .02, color="#BBBBBB", zorder=3,
                   label="shuffled labels" if k == 0 else None)
            ax.errorbar([k - w/2, k + w/2], [r["mean"], s["mean"]],
                        yerr=[r["sem"], s["sem"]], fmt="none", color="#333333",
                        capsize=2, linewidth=.9, zorder=4)
            ticks.append(k); labs.append(c.replace("beta_", "").replace("contrast_value", "contrast"))
    ax.axhline(0, color="#555555", linewidth=1, zorder=2)
    ax.set_xticks(ticks); ax.set_xticklabels(labs, rotation=90, fontsize=7.5, color="#444444")
    for i, mask in enumerate(MASKS):
        ax.text(i * len(cols) + (len(cols) - 1) / 2, ax.get_ylim()[1],
                MASK_LABELS[mask], ha="center", va="bottom", fontsize=9, color="#444444")
    ax.set_ylabel("group mean ± sem", color="#444444")
    ax.set_title("Negative control: shuffled stimulus labels collapse every effect to ~0",
                 fontsize=11, pad=18)
    ax.legend(frameon=False, loc="lower right")
    ax.grid(axis="y", alpha=.25, zorder=0); ax.set_axisbelow(True)
    for s_ in ("top", "right"):
        ax.spines[s_].set_visible(False)
    fig.tight_layout()
    plt.show()

# Unbiasedness + diagonal checks
print("\nmean raw crossnobis distance (rdm_mean) — shuffled should straddle 0:")
for mask in MASKS:
    q = lambda f: f[(f.scope == "pooled") & (f["subset"] == "all") &
                    (f.model == PRIMARY_MODEL) & (f["mask"] == mask)].rdm_mean
    real_m = q(df).mean()
    shuf_m = q(df_shuf).mean() if len(df_shuf) else np.nan
    print(f"  {MASK_LABELS[mask]:14s} real {real_m:+.5f}   shuffled {shuf_m:+.5f}")

diags = [np.abs(np.diag(r)).max() for r in load_rdms(RSA_DIR, "visualcortex", "pooled").values()]
print(f"\nmax |diagonal| over subjects (must be 0): {max(diags) if diags else float('nan')}")

## Findings

_Fill in after running. Each: the number, the n, the test. This section gets promoted into
`session-notes/`._

**Status:** not yet run on the full sample.

1. **Sanity** — β(category) in visual cortex / fusiform:
2. **Value** — β(value) and `contrast_value`, non-figure subset:
3. **Dynamics** — β(value) across learning1 → learning2 → test:
4. **Control** — shuffled-label collapse:

### Open
- Value and frequency are correlated by design (−0.346); with 15–28 cells the partial
  weights are not strongly identified. If both load, a dedicated analysis is needed.
- Extremes of the reward range are not identifiable (the `figure` constraint).
- Searchlight RSA out of scope unless these results warrant it.